main changes:
- updated some variable names for readability across functions
- fixed comparison typo in the PLE function to skip the satisfied clauses
- added early conflict detection in the UP logic, and loop restart when a unit clause is found
- some variable caching, handling empty symbols and conflicts, and nesting removal in DPLL driver function

In [26]:
import json
from pathlib import Path
from argparse import ArgumentParser
from dimacs_parser import DimacsParser
from model_timer import Timer

import numpy as np

# input_file = '../input/C459_4675.cnf'
# input_file = '../input/C1597_081.cnf'
input_file = '../toy_tester.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
print(instance, end="")

Number of variables: 6
Number of clauses: 15
Variables: {1, 2, 3, 4, 5, 6}
Clause 0: {1, 2, 3}
Clause 1: {-1, -2}
Clause 2: {-3, -1}
Clause 3: {-3, -2}
Clause 4: {4, 5, 6}
Clause 5: {-5, -4}
Clause 6: {-6, -4}
Clause 7: {-6, -5}
Clause 8: {1, 4}
Clause 9: {-4, -1}
Clause 10: {2, 5}
Clause 11: {-5, -2}
Clause 12: {2, -6, -1}
Clause 13: {-2, 6}
Clause 14: {1, 6}


In [11]:
symbols = list(instance.vars)
clauses = [list(clause) for clause in instance.clauses]
model = {}

In [12]:
def eval_clause(clause, model):
    unassigned = False 

    for var in clause:
        if (abs(var) in model):
            value = model[abs(var)]

            if (var > 0 and value) or (var < 0 and not value):
                return 'TRUE' 
        else: 
            unassigned = True 
    if unassigned:
        return 'UNKNOWN' 
    
    return 'FALSE'

In [13]:
def eval_instance(clauses, model): 
    every = True 
    for clause in clauses: 
        clause_value = eval_clause(clause, model) 

        if clause_value == 'FALSE': 
            return 'UNSAT' 
        if clause_value != 'TRUE': 
            every = False 
    if every:
        return 'SAT' 
    
    return 'UNKNOWN'

In [ ]:
def pure_symbol(clauses, model):
    pure = {} 
    impure = set()
    for clause in clauses: 
        if eval_clause(clause, model) == 'TRUE': 
            continue 

        for x in clause: 
            var = abs(x)

            if var in model or var in impure:
                continue

            if var not in pure:
                pure[var] = x > 0

            else: 
                if pure[var] != (x > 0): 
                    pure.pop(var)
                    impure.add(var)
    if not pure: 
        return [], []
    return list(pure.keys()), list(pure.values())

In [21]:
# unit clause 
def unit_clause(clauses, model): 
    model = model.copy() 
    new_assignments ={}
    found_unit_clause = 1 

    while found_unit_clause: 
        found_unit_clause = 0 

        for clause in clauses:
            clause_val = eval_clause(clause, model)
            if clause_val == 'TRUE':
                continue 
            elif clause_val == 'FALSE':
                return None, None # there's a conflict. might need to raise
            
            unassigned = [lit for lit in clause if abs(lit) not in model]

            if len(unassigned) == 1: 
                literal = unassigned[0]
                model[abs(literal)] = literal > 0
                new_assignments[abs(literal)] = literal > 0
                found_unit_clause = 1
                break # restart the loop to find potentially new unit clauses


    return list(new_assignments.keys()), list(new_assignments.values())

In [22]:
def dpll(clauses, symbols, model): 
    
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        return True 
    elif instance_status =='UNSAT':
        return False 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return False # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        return dpll(clauses, symbols, model)
    
    # pure literal elimination
    vars, vals = pure_symbol(clauses, model) 
    if vars: 
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        return dpll(clauses, symbols, model)
    
    # branch
    if not symbols:
        return False

    p = symbols[0]
    rest = symbols[1:]

    if dpll(clauses, rest, (model | {p: True})):
        return True 
    
    return dpll(clauses, rest, (model | {p: False})) # backtracking with False

In [27]:
dpll(clauses, symbols, model)

True